In [5]:
import os
import logging

# Enable logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Load Ollama and HuggingFace Embeddings ---
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, ServiceContext, Document
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.indices.postprocessor import SimilarityPostprocessor

# Initialize local LLM
llm = Ollama(model="gemma3:1b", temperature=0.0)
logger.info("Initialized local LLM via Ollama: gemma3:1b")

# Initialize HuggingFace embeddings
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
logger.info("Initialized HuggingFace embeddings")

# Create service context
service_context = ServiceContext.from_defaults(llm=llm, embed_model=embed_model)
logger.info("Created service context")

# --- Load & Parse Web Document ---
import bs4
import requests
from bs4 import BeautifulSoup, SoupStrainer

url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
logger.info(f"Fetching web content from: {url}")

response = requests.get(url)
response.raise_for_status()

strainer = SoupStrainer(class_=("post-content", "post-title", "post-header"))
soup = BeautifulSoup(response.text, "html.parser", parse_only=strainer)
text = soup.get_text(separator="\n").strip()
logger.info(f"Extracted {len(text)} characters of text")

# Convert into LlamaIndex Document
document = Document(text=text)
logger.info("Converted web content to LlamaIndex Document")

# --- Indexing ---
index = VectorStoreIndex.from_documents([document], service_context=service_context)
logger.info("Built VectorStoreIndex")

# --- Query Engine ---
query_engine = index.as_query_engine(similarity_top_k=4)
logger.info("Query engine is ready")



INFO:__main__:Initialized local LLM via Ollama: gemma3:1b
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']
INFO:__main__:Initialized HuggingFace embeddings


ValueError: ServiceContext is deprecated. Use llama_index.settings.Settings instead, or pass in modules to local functions/methods/interfaces.
See the docs for updated usage/migration: 
https://docs.llamaindex.ai/en/stable/module_guides/supporting_modules/service_context_migration/

In [ ]:
# --- Ask a Question ---
question = "What is Task Decomposition?"
logger.info(f"Asking question: {question}")

response = query_engine.query(question)

print("\n=== Answer ===\n", response.response)

# Show partial context (optional)
if response.source_nodes:
    print("\n=== Context Preview ===\n", response.source_nodes[0].node.get_text()[:1000])
else:
    print("\n=== No source context found ===")
